In [1]:
import os
import math
import pandas as pd
from tqdm import tqdm

INPUT_DIR = './data'
OUTPUT_DIR = './output'

def quat_to_euler(w, x, y, z):
    """將四元數轉換為尤拉角 (Roll, Pitch, Yaw)，單位為度"""
    roll = math.atan2(2 * (w * x + y * z), 1 - 2 * (x**2 + y**2))
    sinp = 2 * (w * y - z * x)
    pitch = math.copysign(math.pi / 2, sinp) if abs(sinp) >= 1 else math.asin(sinp)
    yaw = math.atan2(2 * (w * z + x * y), 1 - 2 * (y**2 + z**2))
    return math.degrees(roll), math.degrees(pitch), math.degrees(yaw)

def process_file(infile: str, outfile: str) -> None:
    try:
        df = pd.read_csv(infile, encoding='utf-8')
        if 'data' not in df.columns: 
            return
            
        valid_rows = df['data'].dropna()
        if len(valid_rows) <= 1: 
            return
            
        # 排除第一列雜訊並串接資料
        full_data_str = ''.join(valid_rows.iloc[1:].astype(str))
        
        # 分割資料，並過濾掉因結尾 'end' 產生的純空字串，以確保分母(總列數)計算精準
        raw_segments = [s for s in full_data_str.split('end') if s.strip()]
        total_segments = len(raw_segments)
        
        if total_segments == 0: 
            return

        parsed_data = []
        for segment in raw_segments:
            vals = segment.strip().split('_')
            
            # 條件驗證：長度必須為13
            if len(vals) != 13:
                continue
                
            try:
                # 嘗試全數轉為浮點數，並檢查是否為合法數值 (排除 inf, NaN 等)
                f_vals = [float(v) for v in vals]
                if any(not math.isfinite(v) for v in f_vals):
                    continue
                    
                # 計算尤拉角並組合 17 個欄位 (利用 len(parsed_data) + 1 直接作為完美連續的 ID)
                roll, pitch, yaw = quat_to_euler(*f_vals[9:13])
                parsed_data.append([len(parsed_data) + 1] + f_vals + [roll, pitch, yaw])
                
            except ValueError:
                # 攔截包含 "3.2.2" 等無法轉換為浮點數的異常字串
                continue

        # 計算與輸出統計資訊
        valid_count = len(parsed_data)
        defective_count = total_segments - valid_count
        defective_ratio = (defective_count / total_segments) * 100
        
        # 使用 tqdm.write 避免與進度條的輸出發生畫面衝突
        msg = (f"[{os.path.basename(infile)}] 總數據: {total_segments} 列 | "
               f"正常: {valid_count} 列 | 瑕疵: {defective_count} 列 | "
               f"瑕疵占比: {defective_ratio:.2f}%")
        tqdm.write(msg)

        if not parsed_data:
            return

        # 寫入 CSV
        columns = [
            'ID', 'acceleration_X', 'acceleration_Y', 'acceleration_Z',
            'gyroscope_X', 'gyroscope_Y', 'gyroscope_Z',
            'magnetometer_X', 'magnetometer_Y', 'magnetometer_Z',
            'quaternion_w', 'quaternion_x', 'quaternion_y', 'quaternion_z',
            'euler_roll', 'euler_pitch', 'euler_yaw'
        ]
        pd.DataFrame(parsed_data, columns=columns).to_csv(outfile, index=False, encoding='utf-8')

    except Exception as e:
        tqdm.write(f"處理 {infile} 時發生錯誤: {e}")

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if not os.path.exists(INPUT_DIR):
        print(f"Directory '{INPUT_DIR}' not found.")
        return
        
    files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.csv')]
    for filename in tqdm(files, desc='Processing Files'):
        infile = os.path.join(INPUT_DIR, filename)
        outfile = os.path.join(OUTPUT_DIR, f'{os.path.splitext(filename)[0]}_pre.csv')
        process_file(infile, outfile)

if __name__ == '__main__':
    main()

Processing Files:   0%|                                  | 0/37 [00:00<?, ?it/s]

[1771838245858_nn_160hz_small_esp32_t312_v10_notTired.csv] 總數據: 50436 列 | 正常: 50435 列 | 瑕疵: 1 列 | 瑕疵占比: 0.00%


Processing Files:   3%|▋                         | 1/37 [00:01<00:41,  1.17s/it]

[1771837901232_zhao_160hz_all_esp32_t83_v10_notTired.csv] 總數據: 13493 列 | 正常: 13492 列 | 瑕疵: 1 列 | 瑕疵占比: 0.01%


Processing Files:   5%|█▍                        | 2/37 [00:01<00:22,  1.53it/s]

...

Processing Files:  95%|███████████████████████▋ | 35/37 [00:17<00:01,  1.70it/s]

[1772012111338_zhao_160hz_small_esp32_t123_v10_Tachypnea.csv] 總數據: 20028 列 | 正常: 20027 列 | 瑕疵: 1 列 | 瑕疵占比: 0.00%


Processing Files:  97%|████████████████████████▎| 36/37 [00:18<00:00,  1.86it/s]

[1772013514451_zhao_160hz_small_esp32_t107_v10_notTired.csv] 總數據: 17313 列 | 正常: 17312 列 | 瑕疵: 1 列 | 瑕疵占比: 0.01%


Processing Files: 100%|█████████████████████████| 37/37 [00:18<00:00,  1.99it/s]


In [2]:
import os
import csv

path = './output'
print("-" * 30)

def main(filename):
    data_rows = []
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            reader = csv.reader(file)
            
            # [修正] 跳過標題列，避免將標題計入數據行數
            next(reader, None) 
            
            for row in reader:
                if row:
                    # row[0] 是 ID，我們只取數據部分 row[1:] 進行比對
                    data_rows.append(tuple(row[1:]))
                    #data_rows.append(tuple(row))
    except Exception as e:
        print(f"讀取錯誤 {filename}: {e}")
        return

    if not data_rows:
        return

    L = len(data_rows)
    N = len(set(data_rows))
    
    if L == 0: return

    unique_percentage = N / L * 100

    print(f"檔案: {os.path.basename(filename)}")
    print('相同數據占比: %.2f%%' % (100 - unique_percentage))
    
    try:
        # [確認] 針對 ..._t53_v9_Tired_pre.csv 格式，[-4] 是正確的
        # split結果: [..., 't53', 'v9', 'Tired', 'pre.csv']
        time_part = filename.split('_')[-4]
        
        if time_part.startswith('t'):
            time_val = int(time_part[1:])
            print(f"蒐集頻率: {L/time_val:.2f} Hz")
        else:
            print(f"無法解析時間參數 (預期為 tXX 格式, 讀取到: {time_part})")
            
    except Exception as e:
        print(f"計算頻率錯誤: {e}")
        
    print("-" * 30)

if os.path.exists(path):
    # [優化] 先過濾出 csv 檔並排序，確保取到的是順序正確的最後 5 個
    files = sorted([f for f in os.listdir(path) if f.endswith('.csv')])
    
    for file in files:
        # 簡單過濾檔名特徵
        if 'esp32' in file:
            filename = os.path.join(path, file)
            if os.path.isfile(filename):
                main(filename)
else:
    print(f"找不到路徑: {path}")

------------------------------
檔案: 1771837901232_zhao_160hz_all_esp32_t83_v10_notTired_pre.csv
相同數據占比: 0.24%
蒐集頻率: 162.55 Hz
------------------------------
檔案: 1771837908327_nn_160hz_all_esp32_t94_v10_notTired_pre.csv
相同數據占比: 0.23%
蒐集頻率: 162.55 Hz
------------------------------
檔案: 1771838245295_zhao_160hz_small_esp32_t317_v10_notTired_pre.csv
相同數據占比: 0.21%
蒐集頻率: 161.92 Hz
------------------------------
檔案: 1771838245858_nn_160hz_small_esp32_t312_v10_notTired_pre.csv
相同數據占比: 0.18%
蒐集頻率: 161.65 Hz
------------------------------
檔案: 1771838708821_zhao_160hz_small_esp32_t230_v10_notTired_pre.csv
相同數據占比: 0.22%
蒐集頻率: 162.22 Hz
------------------------------
檔案: 1771838715410_nn_160hz_small_esp32_t240_v10_notTired_pre.csv
相同數據占比: 0.19%
蒐集頻率: 161.61 Hz
------------------------------
檔案: 1771838866705_zhao_160hz_large_esp32_t69_v10_notTired_pre.csv
相同數據占比: 0.20%
蒐集頻率: 163.04 Hz
------------------------------
檔案: 1771838875190_nn_160hz_large_esp32_t74_v10_notTired_pre.csv
相同數據占比: 0.21%
蒐集頻率: 16

In [3]:
import shutil
from pathlib import Path

# 設定來源與目的資料夾
src_dir = Path("output")
dst_dir = Path("output2")

# 若目的資料夾不存在則建立
dst_dir.mkdir(parents=True, exist_ok=True)

# 初始化統計變數
total_files = 0
not_tired_count = 0

for file_path in src_dir.iterdir():
    if file_path.is_file():
        total_files += 1
        
        # 統計檔名中包含 "notTired" 的數量
        if "notTired" in file_path.name:
            not_tired_count += 1
            
        # 重新命名與複製邏輯
        if "Tired" not in file_path.stem:
            new_name = f"{file_path.stem}_Tired{file_path.suffix}"
        else:
            new_name = file_path.name
            
        shutil.copy2(file_path, dst_dir / new_name)

# 輸出統計結果
print("=== 執行完成與統計結果 ===")
if total_files > 0:
    ratio = (not_tired_count / total_files) * 100
    print(f"處理的總檔案數量: {total_files}")
    print(f"檔名含 'notTired' 的數量: {not_tired_count}")
    print(f"'notTired' 檔案佔比: {ratio:.2f}%")
else:
    print("未在 output 資料夾中找到任何檔案，無法計算比例。")

=== 執行完成與統計結果 ===
處理的總檔案數量: 37
檔名含 'notTired' 的數量: 29
'notTired' 檔案佔比: 78.38%


In [4]:
import pandas as pd
import shutil
from pathlib import Path
from sklearn.model_selection import GroupKFold
from tqdm import tqdm

# ======== 參數設定 (依要求調整) ========
IN_DIR = Path("output2")           # 原始資料存放處
OUT_ROOT = Path("K_Fold")       # 最終輸出目錄
WINDOW = 960
STRIDE = 480
K_FOLDS = 5
SEED = 42

def main():
    # 1. 取得原始檔案列表並標記 Group (時間戳記)
    raw_files = sorted(list(IN_DIR.glob("*.csv")))
    if not raw_files:
        print(f"[Error] {IN_DIR} 內找不到 CSV 檔案"); return

    file_meta = []
    for p in raw_files:
        # 依照您的邏輯：取第一個底線前的內容作為 Group ID (如時間戳)
        group_id = p.name.split("_")[0]
        label = "notTired" if "notTired" in p.name else "Tired"
        file_meta.append({"path": p, "label": label, "group": group_id})

    df_meta = pd.DataFrame(file_meta)
    print(f"原始檔案：{len(df_meta)} 個，包含 {df_meta['group'].nunique()} 個獨立 Group。")

    # 2. 準備交叉驗證 (GroupKFold)
    gkf = GroupKFold(n_splits=K_FOLDS)
    
    # 清空輸出目錄
    if OUT_ROOT.exists(): shutil.rmtree(OUT_ROOT)

    # 3. 開始執行 Fold 循環
    # 這裡 split(數據, 標籤, 分組依據)
    for fold, (train_idx, val_idx) in enumerate(gkf.split(df_meta, groups=df_meta["group"]), start=1):
        print(f"\n=== 處理 Fold {fold}/{K_FOLDS} ===")
        
        # 定義這個 Fold 的任務內容
        job_config = {
            "train": df_meta.iloc[train_idx],
            "test":  df_meta.iloc[val_idx]
        }

        for split_name, split_df in job_config.items():
            for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"Processing {split_name}", leave=False):
                # 讀取並切割數據
                df_raw = pd.read_csv(row["path"]).select_dtypes(include=['number'])
                base_name = row["path"].stem
                
                # 滑動視窗切割與儲存
                for start in range(0, len(df_raw) - WINDOW + 1, STRIDE):
                    chunk = df_raw.iloc[start : start + WINDOW]
                    out_filename = f"{base_name}_s{start:06d}.csv"
                    
                    # 建立格式化目錄：K_Fold/fold_1/train/Tired/xxx.csv
                    target_dir = OUT_ROOT / f"fold_{fold}" / split_name / row["label"]
                    target_dir.mkdir(parents=True, exist_ok=True)
                    
                    chunk.to_csv(target_dir / out_filename, index=False)

    print(f"\n✅ 任務完成！輸出資料夾: {OUT_ROOT.resolve()}")

if __name__ == "__main__":
    main()

原始檔案：37 個，包含 37 個獨立 Group。

=== 處理 Fold 1/5 ===



=== 處理 Fold 2/5 ===



=== 處理 Fold 3/5 ===



=== 處理 Fold 4/5 ===



=== 處理 Fold 5/5 ===



✅ 任務完成！輸出資料夾: /home/daen/1150203_Hiking/data_v51/K_Fold
